In [11]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
import time
from datetime import datetime, timedelta
import os

# Initialize storage for article data and Parquet file path
article_data = []
parquet_file = "../../data/00-newspaper_data/crawler/qrohoy/articles.parquet"

# Base URL format for categories
base_url = "https://quintanaroohoy.com/category/{category}/page/{page}/"

# Define categories to scrape
categories = ["mexico", "mundo", "quintanaroo", "seguridad", "100deportes", 
              "like/tecno", "cultura", "voces"]

MAX_RETRIES = 3
MAX_PAGES = 600
def is_relevant_url(url):
    """
    Check if the URL matches the pattern YYYY/MM/DD/title.
    """
    pattern = r'https?://quintanaroohoy\.com/(?!author|category|tag)[a-zA-Z0-9-]+/[a-zA-Z0-9-]+/?$'
    return re.search(pattern, url)


def extract_article_data(url):
    """
    Extract and return the title, main text, date, and source from an article page.
    """
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract the title
        try: 
            title_tag = soup.find('h1', class_='zox-post-title left entry-title')
            title = title_tag.get_text(strip=True) if title_tag else 'No title found'
        except:
            title = None

        # Extract the main text
        try:
            dd_content = soup.find_all('div', class_='zox-post-main')
            main_text = ' '.join(p.get_text(strip=True) for div in dd_content for p in div.find_all('p'))
        except:
            main_text = None

        try:
            subtitle_tag = soup.find('span', class_='zox-post-excerpt')
            subtitle = ' '.join(p.get_text(strip=True) for div in subtitle_tag for p in div.find_all('p'))
        except: 
            subtitle = None

        try:
            date_tag = soup.find('time')
            date = date_tag['datetime'] if date_tag and 'datetime' in date_tag.attrs else 'No date found'
        except: 
            date = None
        # Extract the source

        try:
            category_tag = soup.find('span', class_='zox-post-cat')
            category = category_tag.get_text(strip=True) if title_tag else 'No category found'
        except: 
            category = None

        return {
            'url': url,
            'title': title,
            'sub_title':subtitle,
            'main_text': main_text.strip(),
            'date': date,
            'topic': category
        }
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {url}: {e}")
        return None


def crawl_category(category):
    """
    Crawl pages 1 to 600 in a given category, extracting article links and data.
    """
    for page in range(1, MAX_PAGES + 1):
        category_url = base_url.format(category=category, page=page)
        print(f"Accessing category page: {category_url}")

        retries = 0
        while retries < MAX_RETRIES:
            try:
                response = requests.get(category_url, timeout=15)
                if response.status_code != 200:
                    print(f"Page {page} does not exist in category {category}. Moving to next page.")
                    break  # Stop retrying and move to the next page
                
                soup = BeautifulSoup(response.text, 'html.parser')

                # Find and process all article links on the page
                unique_links = set()
                for link in soup.find_all("a", href=True):
                    href = urljoin(category_url, link['href'])
                    
                    if is_relevant_url(href):
                        unique_links.add(href)  # Store unique article URLs

                # Extract article data immediately
                category_data = []
                for link in unique_links:
                    article = extract_article_data(link)
                    if article:
                        category_data.append(article)
                        print(f"Extracted article from {link}")

                # Save to Parquet
                if category_data:
                    save_to_parquet(category_data, parquet_file)

                time.sleep(2)  # Avoid overloading the server
                break  # If the request was successful, break the retry loop

            except requests.exceptions.RequestException as e:
                retries += 1
                print(f"Connection error on {category_url}. Retrying ({retries}/{MAX_RETRIES})...")
                time.sleep(5)  # Wait before retrying
        
        if retries == MAX_RETRIES:
            print(f"Skipping page {page} in category {category} after {MAX_RETRIES} failed attempts.")
            continue  # Move to the next page even if this one failed


# Function to save data to Parquet
def save_to_parquet(data, parquet_file):
    df = pd.DataFrame(data)
    if not df.empty:
        if os.path.exists(parquet_file):
            initial = pd.read_parquet(parquet_file)
            df = pd.concat([initial, df]).reset_index(drop=True)
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")
        else:
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")


# Crawl all categories and extract data
for category in categories:
    print(f"Starting crawl for category: {category}")
    crawl_category(category)

print("Crawling completed. Data saved in Parquet format.")


Starting crawl for category: mexico
Accessing category page: https://quintanaroohoy.com/category/mexico/page/1/
Extracted article from http://quintanaroohoy.com/mexico/mexico-listo-para-enfrentar-cualquier-arancel-asegura-sheinbaum/
Extracted article from http://quintanaroohoy.com/mexico/presentan-ley-contra-burocracia-y-corrupcion/
Extracted article from http://quintanaroohoy.com/mexico/detienen-en-cdmx-a-joel-medina-la-morsa-colaborador-de-aureliano-guzman-loera/
Extracted article from http://quintanaroohoy.com/entretenimiento/karla-sofia-gascon-de-masterchef-celebrity-a-la-historia-de-los-oscars/
Extracted article from http://quintanaroohoy.com/mexico/profeco-avanza-alerta-por-riesgo-de-incendio-en-miles-de-autos-hyundai/
Extracted article from http://quintanaroohoy.com/mexico/sheinbaum-impulsa-reforma-para-defender-el-maiz-mexicano-de-los-transgenicos/
Extracted article from http://quintanaroohoy.com/mexico/sheinbaum-anuncia-35000-empleos-para-deportados/
Extracted article from htt